In [1]:
import pandas as pd
import numpy as np

# Date pivot pour l'analyse : Mardi 10 Mars 2026
print("Initialisation du pipeline ETL...")

Initialisation du pipeline ETL...


In [2]:
# Chargement du fichier original (Bronze)
df_trans = pd.read_csv('transactions.csv')

# 1. Vérification des identifiants (Attendu : TRX_) [cite: 503]
mask_id_err = ~df_trans['transaction_id'].astype(str).str.startswith('TRX_')
print(f"IDs mal formatés : {mask_id_err.sum()}")

# 2. Valeurs manquantes critiques (Clés Primaires/Étrangères) [cite: 480, 503]
missing_critique = df_trans[['transaction_id', 'customer_id', 'product_id']].isnull().sum()
print(f"Manquants critiques :\n{missing_critique}")

# 3. Validation des types de dates
df_trans['purchase_date'] = pd.to_datetime(df_trans['purchase_date'], errors='coerce')
df_trans['actual_delivery_date'] = pd.to_datetime(df_trans['actual_delivery_date'], errors='coerce')
print(f"Erreurs de format date : {df_trans['purchase_date'].isna().sum()}")

IDs mal formatés : 67164
Manquants critiques :
transaction_id    0
customer_id       0
product_id        0
dtype: int64
Erreurs de format date : 0


In [3]:
# 1. Erreur Logique : Livraison avant l'achat [cite: 503]
mask_date_logic = df_trans['actual_delivery_date'] < df_trans['purchase_date']
print(f"Anomalies 'Livraison < Achat' : {mask_date_logic.sum()}")

# 2. Contraintes Valeurs : Quantité > 0 et Prix >= 0 [cite: 503]
mask_val_err = (df_trans['quantity'] <= 0) | (df_trans['total_price'] < 0)
print(f"Valeurs quantitatives invalides : {mask_val_err.sum()}")

# 3. Outliers de Prix : Détection via le Z-score 
z_scores = (df_trans['total_price'] - df_trans['total_price'].mean()) / df_trans['total_price'].std()
mask_outliers = np.abs(z_scores) > 3
print(f"Outliers de prix (Z-score > 3) : {mask_outliers.sum()}")

Anomalies 'Livraison < Achat' : 66726
Valeurs quantitatives invalides : 121889
Outliers de prix (Z-score > 3) : 31064


In [4]:
# Stratégie : Supprimer les erreurs critiques, Caper les outliers [cite: 241, 242]
idx_drop = df_trans[mask_id_err | df_trans['transaction_id'].isna() | mask_date_logic | mask_val_err].index
df_trans_silver = df_trans.drop(idx_drop).copy()

# Remplacement des outliers par la médiane
mediane_prix = df_trans_silver['total_price'].median()
df_trans_silver.loc[mask_outliers, 'total_price'] = mediane_prix

# Exportation
df_trans_silver.to_csv('transactions_silver.csv', index=False)
print(f"Nouveau fichier 'transactions_silver.csv' créé ({len(df_trans_silver)} lignes).")

Nouveau fichier 'transactions_silver.csv' créé (1795492 lignes).


In [5]:
# Chargement original
df_acc = pd.read_csv('accounting.csv')

# 1. Vérification ID (Attendu : ACC_) [cite: 555]
mask_acc_id = ~df_acc['expense_id'].astype(str).str.startswith('ACC_')
print(f"IDs mal formatés : {mask_acc_id.sum()}")

# 2. Format de Période Fiscale (Attendu : YYYY-QN) [cite: 555, 638]
mask_fiscal = ~df_acc['fiscal_period'].str.contains(r'^\d{4}-Q[1-4]$', na=False)
print(f"Périodes fiscales invalides : {mask_fiscal.sum()}")

IDs mal formatés : 763
Périodes fiscales invalides : 761


In [6]:
# 1. Montants négatifs (Interdit : amount >= 0) [cite: 555]
mask_acc_neg = df_acc['amount'] < 0
print(f"Dépenses avec montants négatifs : {mask_acc_neg.sum()}")

Dépenses avec montants négatifs : 748


In [7]:
# Stratégie : Supprimer les montants impossibles, marquer 'Unknown' pour le flou [cite: 241]
df_acc_silver = df_acc[~(mask_acc_id | mask_acc_neg | df_acc['expense_id'].isna())].copy()

# Conservation de la valeur financière en cas de période mal formatée
df_acc_silver.loc[mask_fiscal, 'fiscal_period'] = 'UNKNOWN_PERIOD'

# Exportation
df_acc_silver.to_csv('accounting_silver.csv', index=False)
print(f"Nouveau fichier 'accounting_silver.csv' créé.")

Nouveau fichier 'accounting_silver.csv' créé.


In [8]:
# Chargement original
df_carb = pd.read_csv('carbon_regulations.csv')

# 1. Contraintes temporelles et tarifaires [cite: 592]
mask_carb_year = df_carb['year'] < 2020
mask_carb_price = df_carb['co2_price_per_ton'] < 0
mask_carb_id = ~df_carb['regulation_id'].astype(str).str.startswith('REG_')

print(f"Années invalides (< 2020) : {mask_carb_year.sum()}")
print(f"Prix CO2 négatifs : {mask_carb_price.sum()}")

Années invalides (< 2020) : 0
Prix CO2 négatifs : 0


In [9]:
# Nettoyage par exclusion
df_carb_silver = df_carb[~(mask_carb_year | mask_carb_price | mask_carb_id)].copy()

# Exportation vers le Data Warehouse
df_carb_silver.to_csv('carbon_regulations_silver.csv', index=False)
print(f"Nouveau fichier 'carbon_regulations_silver.csv' créé.")

Nouveau fichier 'carbon_regulations_silver.csv' créé.


In [10]:
# Synthèse du passage Bronze -> Silver
recap = pd.DataFrame({
    'Table': ['Transactions', 'Accounting', 'Carbon'],
    'Initial (Bronze)': [len(df_trans), len(df_acc), len(df_carb)],
    'Nettoyé (Silver)': [len(df_trans_silver), len(df_acc_silver), len(df_carb_silver)]
})
recap['Taux de Qualité (%)'] = round((recap['Nettoyé (Silver)'] / recap['Initial (Bronze)']) * 100, 2)
recap

,Table,Initial (Bronze),Nettoyé (Silver),Taux de Qualité (%)
0,Transactions,2000000,1795492,89.77
1,Accounting,15000,13695,91.30
2,Carbon,30,30,100.00


In [11]:
# Chargement de la source originale
df_targets = pd.read_csv('sales_targets.csv')

print(f"Nombre de lignes d'objectifs : {len(df_targets)}")
df_targets.head()

Nombre de lignes d'objectifs : 2800


,target_id,store_id,period,target_type,target_value,achievement
0,TGT_f7275c75,MAG_6bf2f675,2017-Q4,nombre_transactions,4387.00,5679.00
1,TGT_b430a744,MAG_4456ebbd,2024-Q4,panier_moyen,37.44,40.14
2,BADID,MAG_UNKNOWN,2018-Q4,nombre_transactions,7334.00,5347.00
3,TGT_00a0629e,MAG_00d620f0,2019-Q2,panier_moyen,15.03,12.76
4,TGT_a50bd4aa,MAG_4c2daf14,2020-Q3,chiffre_affaires,668312.00,509313.00


In [12]:
# 1. Vérification de l'ID (Doit commencer par 'TGT_') [cite: 578]
mask_tgt_id = ~df_targets['target_id'].astype(str).str.startswith('TGT_')
print(f"IDs mal formatés : {mask_tgt_id.sum()}")

# 2. Valeurs manquantes critiques (Clés Primaires et Étrangères)
# Une cible sans ID ou sans magasin associé est inutile pour la BI
missing_targets = df_targets[['target_id', 'store_id']].isnull().sum()
print(f"\nManquants sur les clés :\n{missing_targets}")

# 3. Validation du format de la période (YYYY-QN ou YYYY-MM) [cite: 578]
# On vérifie si le format respecte la norme pour les jointures temporelles
mask_period_err = ~df_targets['period'].str.contains(r'^\d{4}-(Q[1-4]|0[1-9]|1[0-2])$', na=False)
print(f"Périodes mal formatées : {mask_period_err.sum()}")

IDs mal formatés : 167

Manquants sur les clés :
target_id    0
store_id     0
dtype: int64
Périodes mal formatées : 170


C:\Users\bouss\AppData\Local\Temp\ipykernel_12812\3001521478.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_period_err = ~df_targets['period'].str.contains(r'^\d{4}-(Q[1-4]|0[1-9]|1[0-2])$', na=False)


In [13]:
# 1. Vérification du type d'objectif [cite: 578]
# Doit être : chiffre_affaires / nombre_transactions / panier_moyen / taux_retour
valid_types = ['chiffre_affaires', 'nombre_transactions', 'panier_moyen', 'taux_retour']
mask_type_err = ~df_targets['target_type'].isin(valid_types)
print(f"Types d'objectifs inconnus : {mask_type_err.sum()}")

# 2. Valeurs négatives (Interdit par le catalogue : >= 0) [cite: 578]
# Un objectif ou une réalisation ne peut pas être négatif
mask_neg_val = (df_targets['target_value'] < 0) | (df_targets['achievement'] < 0)
print(f"Valeurs négatives détectées : {mask_neg_val.sum()}")

# 3. Calcul du taux de réalisation pour détecter des outliers massifs
# Une réalisation de 1000% est suspecte et pourrait être une erreur de saisie
df_targets['achievement_rate'] = df_targets['achievement'] / df_targets['target_value']
mask_target_outliers = df_targets['achievement_rate'] > 5  # Plus de 500% de l'objectif
print(f"Réalisations suspectes (>500%) : {mask_target_outliers.sum()}")

Types d'objectifs inconnus : 0
Valeurs négatives détectées : 277
Réalisations suspectes (>500%) : 8


In [14]:
# Stratégie de nettoyage :
# A. SUPPRESSION : On élimine les IDs invalides, les manquants critiques et les valeurs négatives
idx_to_drop_tgt = df_targets[mask_tgt_id | df_targets['target_id'].isna() | mask_neg_val].index
df_targets_silver = df_targets.drop(idx_to_drop_tgt).copy()

# B. MARQUAGE : Pour les types d'objectifs inconnus, on les classe en 'autre' pour ne pas perdre la donnée
df_targets_silver.loc[mask_type_err, 'target_type'] = 'autre'

# C. EXPORTATION
df_targets_silver.to_csv('sales_targets_silver.csv', index=False)

print(f"--- RÉSULTAT FINAL ---")
print(f"Lignes conservées : {len(df_targets_silver)}")
print(f"Fichier 'sales_targets_silver.csv' généré avec succès.")

--- RÉSULTAT FINAL ---
Lignes conservées : 2457
Fichier 'sales_targets_silver.csv' généré avec succès.
